In [0]:
# Import LogControl class from parent notebook
# This runs the logger_control notebook and makes the LogControl class available

In [0]:
%run "../src/logger_control"

# LogControl Test Suite

## Overview

This notebook contains comprehensive tests for the **LogControl** class.

## Test Coverage

### Tests Included:

1. **Initialization Tests (2)**
   * Default parameters
   * Custom parameters

2. **Log Level Methods (4)**
   * `log_info()` - INFO messages
   * `log_success()` - SUCCESS messages  
   * `log_warning()` - WARNING messages
   * `log_error()` - ERROR messages

3. **Error Handler (3)**
   * ZeroDivisionError with stack trace
   * ValueError capture
   * KeyError capture

4. **DataFrame Creation (1)**
   * None handling

5. **Edge Cases (3)**
   * Special characters (\n, \t, quotes, etc.)
   * Unicode support (你好, мир, 🎉)
   * Multiple independent logger instances

---

## Running Tests

**Execute Cell 3** to run all tests.

* **Total**: 13 tests
* **Expected**: 100% passing
* **Duration**: ~1 second

---

## Location

```
/data-in-code/vendas_regionais/sdd/features/error_handler_logging/
├── logger_control (main notebook with LogControl class)
└── src/tests/
    └── test_logger_control (this notebook)
```

In [0]:
# Complete Test Suite for LogControl
# Executes all tests and reports results

import logging
from io import StringIO

class TestRunner:
    """Simple test runner."""
    def __init__(self):
        self.passed = 0
        self.failed = 0
        self.errors = []
    
    def test(self, name, func):
        try:
            func()
            self.passed += 1
            print(f"  ✓ {name}")
        except AssertionError as e:
            self.failed += 1
            self.errors.append((name, str(e)))
            print(f"  ✗ {name}: {e}")
        except Exception as e:
            self.failed += 1
            self.errors.append((name, f"Error: {e}"))
            print(f"  ✗ {name}: {e}")
    
    def summary(self):
        print("\n" + "="*70)
        print(f"Results: {self.passed} passed, {self.failed} failed")
        print("="*70)
        if self.failed == 0:
            print("\n✅ All tests passed!")
        else:
            print(f"\n❌ {self.failed} test(s) failed")
            for name, error in self.errors:
                print(f"  - {name}: {error}")

runner = TestRunner()

print("\n" + "="*70)
print("LogControl Test Suite")
print("="*70)

# Test 1: Initialization
print("\n[Initialization Tests]")
def test_init():
    logger = LogControl(logger_name="test1", tbl_name="test.logs")
    assert logger._logger is not None
    assert logger.tbl_name == "test.logs"
    for h in logger._logger.handlers[:]: logger._logger.removeHandler(h)
runner.test("test_initialization", test_init)

def test_init_defaults():
    logger = LogControl()
    assert logger._logger.name == "logcontrol_log_cls"
    assert logger.tbl_name is None
    for h in logger._logger.handlers[:]: logger._logger.removeHandler(h)
runner.test("test_initialization_defaults", test_init_defaults)

# Test 2-5: Log Level Methods
print("\n[Log Level Tests]")
def test_log_info():
    logger = LogControl(logger_name="test_info")
    stream = StringIO()
    handler = logging.StreamHandler(stream)
    logger._logger.addHandler(handler)
    logger.log_info("Test message")
    output = stream.getvalue()
    assert "[INFO] Test message" in output
    for h in logger._logger.handlers[:]: logger._logger.removeHandler(h)
runner.test("test_log_info", test_log_info)

def test_log_success():
    logger = LogControl(logger_name="test_success")
    stream = StringIO()
    handler = logging.StreamHandler(stream)
    logger._logger.addHandler(handler)
    logger.log_success("Success message")
    output = stream.getvalue()
    assert "[SUCCESS] Success message" in output
    for h in logger._logger.handlers[:]: logger._logger.removeHandler(h)
runner.test("test_log_success", test_log_success)

def test_log_warning():
    logger = LogControl(logger_name="test_warning")
    stream = StringIO()
    handler = logging.StreamHandler(stream)
    logger._logger.addHandler(handler)
    logger.log_warning("Warning message")
    output = stream.getvalue()
    assert "[WARNING] Warning message" in output
    assert "WARNING" in output
    for h in logger._logger.handlers[:]: logger._logger.removeHandler(h)
runner.test("test_log_warning", test_log_warning)

def test_log_error():
    logger = LogControl(logger_name="test_error")
    stream = StringIO()
    handler = logging.StreamHandler(stream)
    logger._logger.addHandler(handler)
    logger.log_error("Error message")
    output = stream.getvalue()
    assert "[ERROR] Error message" in output
    for h in logger._logger.handlers[:]: logger._logger.removeHandler(h)
runner.test("test_log_error", test_log_error)

# Test 6-8: Error Handler
print("\n[Error Handler Tests]")
def test_error_handler_console():
    logger = LogControl(logger_name="test_eh")
    stream = StringIO()
    handler = logging.StreamHandler(stream)
    logger._logger.addHandler(handler)
    try:
        x = 1 / 0
    except Exception as e:
        result = logger.error_handler(e, debug_write_mode=False)
        assert result is None
    output = stream.getvalue()
    assert "ZeroDivisionError" in output
    for h in logger._logger.handlers[:]: logger._logger.removeHandler(h)
runner.test("test_error_handler_console_only", test_error_handler_console)

def test_error_handler_valueerror():
    logger = LogControl(logger_name="test_ve")
    stream = StringIO()
    handler = logging.StreamHandler(stream)
    logger._logger.addHandler(handler)
    try:
        raise ValueError("Test error")
    except Exception as e:
        logger.error_handler(e, debug_write_mode=False)
    output = stream.getvalue()
    assert "ValueError" in output
    assert "Test error" in output
    for h in logger._logger.handlers[:]: logger._logger.removeHandler(h)
runner.test("test_error_handler_captures_valueerror", test_error_handler_valueerror)

def test_error_handler_keyerror():
    logger = LogControl(logger_name="test_ke")
    stream = StringIO()
    handler = logging.StreamHandler(stream)
    logger._logger.addHandler(handler)
    try:
        data = {"key": "value"}
        _ = data["missing"]
    except Exception as e:
        logger.error_handler(e, debug_write_mode=False)
    output = stream.getvalue()
    assert "KeyError" in output
    for h in logger._logger.handlers[:]: logger._logger.removeHandler(h)
runner.test("test_error_handler_captures_keyerror", test_error_handler_keyerror)

# Test 9: DataFrame Creation
print("\n[DataFrame Creation Tests]")
def test_create_dataframe_none():
    logger = LogControl(logger_name="test_df")
    result = logger.create_dataframe(None)
    assert result is None
    for h in logger._logger.handlers[:]: logger._logger.removeHandler(h)
runner.test("test_create_dataframe_with_none", test_create_dataframe_none)

# Test 10-12: Edge Cases
print("\n[Edge Case Tests]")
def test_special_chars():
    logger = LogControl(logger_name="test_special")
    stream = StringIO()
    handler = logging.StreamHandler(stream)
    logger._logger.addHandler(handler)
    msg = "Test: \n\t' \" < >"
    logger.log_info(msg)
    output = stream.getvalue()
    assert msg in output
    for h in logger._logger.handlers[:]: logger._logger.removeHandler(h)
runner.test("test_log_with_special_characters", test_special_chars)

def test_unicode():
    logger = LogControl(logger_name="test_unicode")
    stream = StringIO()
    handler = logging.StreamHandler(stream)
    logger._logger.addHandler(handler)
    msg = "Test: 你好 мир 🎉"
    logger.log_info(msg)
    output = stream.getvalue()
    assert msg in output
    for h in logger._logger.handlers[:]: logger._logger.removeHandler(h)
runner.test("test_log_with_unicode", test_unicode)

def test_multiple_loggers():
    logger1 = LogControl(logger_name="logger1")
    logger2 = LogControl(logger_name="logger2")
    assert logger1._logger.name != logger2._logger.name
    for h in logger1._logger.handlers[:]: logger1._logger.removeHandler(h)
    for h in logger2._logger.handlers[:]: logger2._logger.removeHandler(h)
runner.test("test_multiple_loggers_independent", test_multiple_loggers)

# Print Summary
runner.summary()